# Criterion 2 — Supervised Learning and Classification

## Feature Engineering — Creating Derived Features

Raw STATS19 fields like hour (0–23) and day_of_week (1–7) contain useful signal but can be noisy if fed directly into a model. I will derive three higher-level features to help the models learn broader patterns:

| Feature | Rationale |
|---|---|
| is_weekend | Weekend driving behaviour (leisure trips, late-night driving) differs from weekday commuting patterns. Encoding Saturday/Sunday as a binary flag is more interpretable than leaving day_of_week as a raw integer. |
| time_of_day_bin | Grouping hours into four bins (Night / Morning / Afternoon / Evening) reduces noise. Individual hours are sparse — there are roughly 10× more collisions in the afternoon peak than at 3 am — so binning helps tree models find a clean split without overfitting to one specific hour. |
| is_dark | STATS19 light_conditions codes 4 and 5 both represent darkness (with and without street lighting). Merging them into a single binary feature simplifies the signal: darkness vs. not darkness. |

We also label-encode six categorical columns so that all classifiers receive numeric input.

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/Preprocessed_Traffic_Data.csv", low_memory=False)

# is_weekend: 1=Sunday, 7=Saturday in STATS19
df["is_weekend"] = df["day_of_week"].isin([1, 7]).astype(int)

# time_of_day_bin: 0=Night, 1=Morning, 2=Afternoon, 3=Evening
df["time_of_day_bin"] = pd.cut(df["hour"],
    bins=[-1, 6, 12, 18, 24],
    labels=[0, 1, 2, 3]).astype(int)

# is_dark: light_conditions 4 = dark, 5 = dark with street lights
df["is_dark"] = df["light_conditions"].isin([4, 5]).astype(int)

# Label-encode categorical features
cat_features = ["road_type", "weather_conditions", "road_surface_conditions",
                "junction_detail", "junction_control", "urban_or_rural_area"]
le = LabelEncoder()
for col in cat_features:
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))

## Define Feature Sets and Train / Val / Test Split

I used a strict 60% train / 15% validation / 25% test split to:
- Train on the majority of data,
- Tune hyperparameters on the validation set,
- Report final, unbiased metrics on the held-out test set.

In [2]:
from sklearn.model_selection import train_test_split

FEATURES_A = ["weather_conditions_enc", "road_type_enc", "light_conditions",
              "speed_limit", "number_of_vehicles", "road_surface_conditions_enc",
              "junction_detail_enc", "junction_control_enc", "urban_or_rural_area_enc",
              "day_of_week", "hour", "is_weekend", "time_of_day_bin", "is_dark"]

X_a = df[FEATURES_A]
y_a = df["collision_severity"]

# First split: 75% train+val, 25% test
X_tv, X_test, y_tv, y_test = train_test_split(
    X_a, y_a, test_size=0.25, random_state=42, stratify=y_a)

# Second split: 60% train, 15% val
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.20, random_state=42, stratify=y_tv)

print(f"Train: {len(X_train)} ({len(X_train)/len(X_a)*100:.0f}%)")
print(f"Val:   {len(X_val)}  ({len(X_val)/len(X_a)*100:.0f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X_a)*100:.0f}%)")

Train: 4759 (60%)
Val:   1190  (15%)
Test:  1984 (25%)


## Handle Class Imbalance with SMOTE

SMOTE (Synthetic Minority Over-sampling Technique) is applied only to the training set. Applying it to validation or test data would be data leakage as the model would be evaluated on artificially generated samples rather than real-world distributions which could skew results.

In [3]:
from imblearn.over_sampling import SMOTE

print("BEFORE SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nAFTER SMOTE:")
print(pd.Series(y_train_res).value_counts())

BEFORE SMOTE:
collision_severity
3.0    3930
2.0     770
1.0      59
Name: count, dtype: int64

AFTER SMOTE:
collision_severity
2.0    3930
3.0    3930
1.0    3930
Name: count, dtype: int64


## Task A — Multiclass Classification: Collision Severity (Fatal / Serious / Slight)

We train 5 classifiers on the SMOTE-balanced training data and compare validation accuracy.

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

classifiers = {
    "KNN":                  KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)":            SVC(kernel="rbf", class_weight="balanced", random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "Logistic Regression":  LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree":        DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Naive Bayes":          GaussianNB(),
    "KNN (k=11)":           KNeighborsClassifier(n_neighbors=11)
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train_res, y_train_res)
    y_pred_val = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred_val)
    results[name] = acc
    print(f"{name}: Validation Accuracy = {acc:.4f}")

KNN: Validation Accuracy = 0.5815
SVM (RBF): Validation Accuracy = 0.5445
Random Forest: Validation Accuracy = 0.7445
Logistic Regression: Validation Accuracy = 0.5529
Decision Tree: Validation Accuracy = 0.6899
Gradient Boosting: Validation Accuracy = 0.7454
Naive Bayes: Validation Accuracy = 0.4403
KNN (k=11): Validation Accuracy = 0.5319


## Hyperparameter Tuning — Random Forest

GridSearchCV uses 5-fold cross-validation on the training data to find the best hyperparameters. We then evaluate the final model on the held-out test set.

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    "n_estimators":     [100, 200, 300],
    "max_depth":        [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid_rf, cv=5, scoring="f1_macro", n_jobs=-1
)
grid_rf.fit(X_train_res, y_train_res)

print("Best RF params:", grid_rf.best_params_)
print("Best CV F1:    ", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_
y_pred_test = best_rf.predict(X_test)
print("\n--- Test Set Report (Multiclass) ---")
print(classification_report(y_test, y_pred_test,
      target_names=["Fatal", "Serious", "Slight"]))

Best RF params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Best CV F1:     0.8896099299724259

--- Test Set Report (Multiclass) ---
              precision    recall  f1-score   support

       Fatal       0.05      0.04      0.05        25
     Serious       0.26      0.18      0.21       321
      Slight       0.84      0.89      0.86      1638

    accuracy                           0.77      1984
   macro avg       0.38      0.37      0.37      1984
weighted avg       0.73      0.77      0.75      1984



## Task B — Binary Classification: Is this collision Fatal or Not?

We collapse the three severity classes into a simpler binary problem: Fatal (1) vs. Not Fatal (0). This is useful for prioritising emergency response resources.

In [6]:
# Build binary target
y_binary = (df["collision_severity"] == 1).astype(int)
X_b = df[FEATURES_A]

# Same 60/15/25 split
Xb_tv, Xb_test, yb_tv, yb_test = train_test_split(
    X_b, y_binary, test_size=0.25, random_state=42, stratify=y_binary)
Xb_train, Xb_val, yb_train, yb_val = train_test_split(
    Xb_tv, yb_tv, test_size=0.20, random_state=42, stratify=yb_tv)

# SMOTE for binary imbalance
Xb_train_res, yb_train_res = SMOTE(random_state=42).fit_resample(Xb_train, yb_train)

print("Binary class distribution after SMOTE:")
print(pd.Series(yb_train_res).value_counts())

# Train a Logistic Regression and a Random Forest
binary_classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
}

for name, clf in binary_classifiers.items():
    clf.fit(Xb_train_res, yb_train_res)
    y_pred = clf.predict(Xb_test)
    print(f"\n--- {name} (Binary) ---")
    print(classification_report(yb_test, y_pred, target_names=["Not Fatal", "Fatal"]))

Binary class distribution after SMOTE:
collision_severity
0    4700
1    4700
Name: count, dtype: int64

--- Logistic Regression (Binary) ---
              precision    recall  f1-score   support

   Not Fatal       0.99      0.80      0.89      1959
       Fatal       0.02      0.32      0.04        25

    accuracy                           0.80      1984
   macro avg       0.50      0.56      0.46      1984
weighted avg       0.98      0.80      0.88      1984


--- Random Forest (Binary) ---
              precision    recall  f1-score   support

   Not Fatal       0.99      1.00      0.99      1959
       Fatal       0.00      0.00      0.00        25

    accuracy                           0.99      1984
   macro avg       0.49      0.50      0.50      1984
weighted avg       0.97      0.99      0.98      1984



## Task C — Categorical Classification: Urban vs Rural

We predict whether a collision occurred in an urban or rural area. This is a binary classification, but it tests a completely different target variable.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# filter out missing records to ensure a strict binary classification
valid_indices = df["urban_or_rural_area"].isin([1, 2, '1', '2'])
df_c = df[valid_indices].copy()

FEATURES_C = ["weather_conditions_enc", "road_type_enc", "light_conditions",
              "speed_limit", "number_of_vehicles", "road_surface_conditions_enc",
              "junction_detail_enc", "junction_control_enc",
              "day_of_week", "hour", "is_weekend", "time_of_day_bin", "is_dark"]

X_c = df_c[FEATURES_C]

# 1 = ubran, 2 = rural
y_c = (df_c["urban_or_rural_area"].astype(int) == 1).astype(int)

# create the splits on the filtered data.
Xc_tv, Xc_test, yc_tv, yc_test = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c)
Xc_train, Xc_val, yc_train, yc_val = train_test_split(
    Xc_tv, yc_tv, test_size=0.20, random_state=42, stratify=yc_tv)

# apply SMOTE to the training set
Xc_train_res, yc_train_res = SMOTE(random_state=42).fit_resample(Xc_train, yc_train)

# train and then evaluate the model
rf_urban = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf_urban.fit(Xc_train_res, yc_train_res)
yc_pred = rf_urban.predict(Xc_test)

print("--- Urban vs Rural Classification (Random Forest) ---")
print(classification_report(yc_test, yc_pred, target_names=["Rural", "Urban"]))

--- Urban vs Rural Classification (Random Forest) ---
              precision    recall  f1-score   support

       Rural       0.54      0.52      0.53       103
       Urban       0.95      0.96      0.96      1082

    accuracy                           0.92      1185
   macro avg       0.75      0.74      0.74      1185
weighted avg       0.92      0.92      0.92      1185



## Responsible AI Considerations

### Class Imbalance

The STATS19 dataset is heavily skewed towards *Slight* collisions (roughly 79% of all records), with *Fatal* collisions representing only about 1.2%. Without correction, any classifier will achieve high overall accuracy simply by predicting *Slight* for every row. We addressed this using SMOTE on the training set and class_weight="balanced where available. However, it is important to note:

- SMOTE generates synthetic samples by interpolating between existing minority-class examples. These are approximations and not real collision events.
- The macro-average F1 is a more honest performance metric than accuracy for this imbalanced problem.

### Bias Risks

- Reporting bias: STATS19 only captures collisions reported to police. Minor incidents where nobody called for help are absent from the dataset, likely underrepresenting mild injury events.
- Temporal bias: Road conditions, vehicle safety technology, and speed limits have all changed significantly between 1979 and 2020. A model trained on the full date range may apply patterns from older records that are no longer valid.
- Geographic bias: This dataset covers Sheffield only. Models trained here may not generalise to rural counties, motorway networks, or other cities with different road infrastructure.
- Feature proxy risk: Features like speed_limit and urban_or_rural_area may act as proxies for socioeconomic deprivation in certain areas. Models making high-severity predictions disproportionately for deprived areas could unfairly influence resource allocation decisions.

### Temporal Limits

This data spans between 1979 - 2020, the models may be skewed due to the fact that the sharp drop of was mainly caused by COVID lockdowns which dramatically decreased the collisions for that period, I would expect that in an updated dataset the collisions would actually rise gradually after 2020.

In [8]:
df.to_csv("../data/Engineered_Traffic_Data.csv", index=False)
print("Saved Engineered_Traffic_Data.csv")

Saved Engineered_Traffic_Data.csv
